# Top ddE double-mutation pairs: dE / ddE table — **IN on the NL4-3 background**

Same calculation as `top_20_DRM_dde.ipynb`, except **IN uses the NL4-3 strain
sequence as the background instead of the subtype-B consensus**. PR and RT are
unchanged (still on their consensus).

Reads `{protein}/data/top_20_DRM.txt` and computes, on the chosen background:

| column | meaning |
|---|---|
| `dE_m1`  | delta E of the first single mutation |
| `dE_m2`  | delta E of the second single mutation |
| `dE_dbl` | delta E of the double mutation |
| `ddE`    | `dE_dbl - dE_m1 - dE_m2` (epistasis) |

Energies come from `utilities.functions.calculate_dde_v2`
(= `calculate_dde_from_de_with_substitution`), which forces each mutation's own
wildtype residue at its two positions before computing, so only the *rest* of
the background differs between the consensus and NL4-3 runs.

The `background` column records which sequence each row used.

Note: RT's `top_20_DRM.txt` holds 40 pairs — the first 20 are NRTI, the last 20
NNRTI — so RT rows carry a `drug_class` label.

In [1]:
import sys
import importlib
sys.path.append('../')  # repo root

import pandas as pd
import utilities.functions as functions
importlib.reload(functions)

<module 'utilities.functions' from '/Users/xuechenkan/potts_model_test/ms0_5/../utilities/functions.py'>

## Load backgrounds, reduction dictionaries and J matrices

In [2]:
IN_consensus_seq = open('IN/data/in.consensus.reduce4.seq').read().strip()
IN_nl43_seq      = open('IN/data/in.nl43.reduce4.seq').read().strip()
PR_consensus_seq = open('PR/data/pr.consensus.reduce4.seq').read().strip()
RT_consensus_seq = open('RT/data/rt.consensus.reduce4.seq').read().strip()

# IN background for this notebook
IN_background_name = 'NL4-3'
IN_background_seq = IN_nl43_seq

diffs = [(i + 1, c, n) for i, (c, n) in enumerate(zip(IN_consensus_seq, IN_nl43_seq)) if c != n]
print(f'IN consensus vs NL4-3: {len(diffs)} differing reduced positions')
for pos, c, n in diffs:
    print(f'  pos {pos}: consensus {c} -> NL4-3 {n}')

IN consensus vs NL4-3: 4 differing reduced positions
  pos 72: consensus A -> NL4-3 D
  pos 113: consensus C -> NL4-3 D
  pos 151: consensus A -> NL4-3 B
  pos 234: consensus C -> NL4-3 A


In [3]:
IN_min_position, IN_max_position = 1, 263
PR_min_position, PR_max_position = 1, 99
RT_min_position, RT_max_position = 39, 226

IN_redux = functions.get_redu_dict('IN/data/in.reduce4.redux', 1)
PR_redux = functions.get_redu_dict('PR/data/pr.reduce4.redux', 0)
RT_redux = functions.get_redu_dict('RT/data/rt.reduce4.redux', 0)

IN_J = functions.load_J_dict('IN/data/J.npy',    IN_min_position, IN_max_position)
PR_J = functions.load_J_dict('PR/data/J_PR.npy', PR_min_position, PR_max_position)
RT_J = functions.load_J_dict('RT/data/J_RT.npy', RT_min_position, RT_max_position)

proteins = {
    'IN': dict(pairs_file='IN/data/top_20_DRM.txt', background=IN_background_seq,
               background_name=IN_background_name,
               redux=IN_redux, J=IN_J, min_pos=IN_min_position, max_pos=IN_max_position),
    'PR': dict(pairs_file='PR/data/top_20_DRM.txt', background=PR_consensus_seq,
               background_name='consensus',
               redux=PR_redux, J=PR_J, min_pos=PR_min_position, max_pos=PR_max_position),
    'RT': dict(pairs_file='RT/data/top_20_DRM.txt', background=RT_consensus_seq,
               background_name='consensus',
               redux=RT_redux, J=RT_J, min_pos=RT_min_position, max_pos=RT_max_position),
}

## Compute dE_m1, dE_m2, dE_dbl and ddE for every pair

In [4]:
# RT's file is NRTI (first 20) followed by NNRTI (last 20)
RT_N_NRTI = 20


def drug_class(protein, idx):
    if protein != 'RT':
        return {'IN': 'INSTI', 'PR': 'PI'}[protein]
    return 'NRTI' if idx < RT_N_NRTI else 'NNRTI'


def dde_table(protein, cfg):
    rows = []
    pairs = functions.read_list_from_file(cfg['pairs_file'])
    for idx, pair in enumerate(pairs):
        mut1, mut2 = [m.strip() for m in pair.split('-')]
        red1 = functions.unreduced_to_reduced(cfg['redux'], mut1)
        red2 = functions.unreduced_to_reduced(cfg['redux'], mut2)
        if '-' in (red1[0], red1[-1], red2[0], red2[-1]):
            print(f'[{protein}] skipped {mut1}-{mut2}: not in reduction dictionary')
            continue
        dE_m1, dE_m2, dE_dbl, ddE = functions.calculate_dde_v2(
            red1, red2, cfg['background'], cfg['J'], cfg['min_pos'], cfg['max_pos'])
        rows.append({
            'protein': protein,
            'drug_class': drug_class(protein, idx),
            'background': cfg['background_name'],
            'pair': f'{mut1}-{mut2}',
            'mut1': mut1,
            'mut2': mut2,
            'mut1_reduced': red1,
            'mut2_reduced': red2,
            'dE_m1': dE_m1,
            'dE_m2': dE_m2,
            'dE_dbl': dE_dbl,
            'ddE': ddE,
        })
    return pd.DataFrame(rows)


tables = {p: dde_table(p, cfg) for p, cfg in proteins.items()}
df = pd.concat(tables.values(), ignore_index=True)
df

,protein,drug_class,background,pair,mut1,mut2,mut1_reduced,mut2_reduced,dE_m1,dE_m2,dE_dbl,ddE
0,IN,INSTI,NL4-3,G140S-Q148H,G140S,Q148H,C140D,D148B,-6.705915,-4.577813,-2.774946,8.508782
1,IN,INSTI,NL4-3,Y143C-S230R,Y143C,S230R,D143A,D230C,-8.272262,-9.935546,-11.830268,6.377540
2,IN,INSTI,NL4-3,G140A-Q148K,G140A,Q148K,C140A,D148A,-7.272867,-7.578910,-9.238797,5.612979
3,IN,INSTI,NL4-3,G140S-Q148R,G140S,Q148R,C140D,D148C,-6.705915,-4.038246,-5.539972,5.204189
4,IN,INSTI,NL4-3,G140A-Q148R,G140A,Q148R,C140A,D148C,-7.272867,-4.038246,-6.396175,4.914937
...,...,...,...,...,...,...,...,...,...,...,...,...
75,RT,NNRTI,consensus,V108I-V189I,V108I,V189I,B108C,D189A,-4.285306,-5.001471,-8.112434,1.174342
76,RT,NNRTI,consensus,K101E-E138K,K101E,E138K,C101A,C138A,-3.861675,-5.320679,-8.030475,1.151880
77,RT,NNRTI,consensus,E138A-G190E,E138A,G190E,C138B,D190A,-3.707515,-5.147928,-7.760716,1.094728
78,RT,NNRTI,consensus,K101P-D192N,K101P,D192N,C101D,A192D,-5.446384,-6.151916,-10.533675,1.064624


## IN: NL4-3 vs consensus

In [5]:
# Same pairs recomputed on the consensus, to see what the background change does
IN_consensus_cfg = dict(proteins['IN'], background=IN_consensus_seq, background_name='consensus')
IN_on_consensus = dde_table('IN', IN_consensus_cfg)

cmp = tables['IN'][['pair', 'dE_m1', 'dE_m2', 'dE_dbl', 'ddE']].merge(
    IN_on_consensus[['pair', 'dE_m1', 'dE_m2', 'dE_dbl', 'ddE']],
    on='pair', suffixes=('_nl43', '_consensus'))
for col in ['dE_m1', 'dE_m2', 'dE_dbl', 'ddE']:
    cmp[f'{col}_diff'] = cmp[f'{col}_nl43'] - cmp[f'{col}_consensus']
print('max |ddE difference|:', cmp['ddE_diff'].abs().max())
cmp[['pair', 'ddE_consensus', 'ddE_nl43', 'ddE_diff',
     'dE_m1_diff', 'dE_m2_diff', 'dE_dbl_diff']]

max |ddE difference|: 1.7642975e-05


,pair,ddE_consensus,ddE_nl43,ddE_diff,dE_m1_diff,dE_m2_diff,dE_dbl_diff
0,G140S-Q148H,8.508784,8.508782,-1.907349e-06,-0.977237,-0.024308,-1.001547
1,Y143C-S230R,6.377539,6.377540,4.768372e-07,-1.904267,-3.944997,-5.849264
2,G140A-Q148K,5.612984,5.612979,-4.768372e-06,-0.810802,-0.453497,-1.264303
3,G140S-Q148R,5.204190,5.204189,-1.430511e-06,-0.977237,0.115367,-0.861871
4,G140A-Q148R,4.914940,4.914937,-3.337860e-06,-0.810802,0.115367,-0.695437
5,E138K-Q148K,4.900512,4.900515,3.337860e-06,0.470489,-0.453497,0.016995
6,Y143S-S230R,3.700699,3.700700,9.536743e-07,-4.129550,-3.944997,-8.074547
7,E138K-Q148R,3.525786,3.525786,0.000000e+00,0.470489,0.115367,0.585856
8,G140C-Q148R,2.989779,2.989777,-2.384186e-06,-1.707133,0.115367,-1.591768
9,T97A-Y143R,2.836372,2.836373,9.536743e-07,1.445355,-2.253313,-0.807957


## Write the CSVs

In [6]:
for protein, tbl in tables.items():
    out = f'{protein}_top_20_DRM_dde_IN_nl43.csv'
    tbl.to_csv(out, index=False)
    print(f'wrote {out}  ({len(tbl)} pairs, background: {tbl["background"].iloc[0]})')

df.to_csv('top_20_DRM_dde_all_proteins_IN_nl43.csv', index=False)
print(f'wrote top_20_DRM_dde_all_proteins_IN_nl43.csv  ({len(df)} pairs)')

wrote IN_top_20_DRM_dde_IN_nl43.csv  (20 pairs, background: NL4-3)
wrote PR_top_20_DRM_dde_IN_nl43.csv  (20 pairs, background: consensus)
wrote RT_top_20_DRM_dde_IN_nl43.csv  (40 pairs, background: consensus)
wrote top_20_DRM_dde_all_proteins_IN_nl43.csv  (80 pairs)
